# Everything you need to know about RAG

Welcome to the foundation of enterprise AI. Retrieval-Augmented Generation (RAG) mathematically grounds Large Language Models in verifiable facts. 

### The RAG Analytical Pipeline:
1. **Ingestion & Chunking:** Large documents are too big for an LLM's context window, so we slice them into smaller, mathematically overlapping chunks.
2. **Embedding:** We convert human text into high-dimensional vectors (arrays of numbers) representing semantic meaning.
3. **Vector Storage:** We store these vectors in a specialized database optimized for similarity search.
4. **Retrieval:** When a user asks a question, we embed the question and find the closest matching document vectors using Cosine Similarity.
5. **Generation:** We inject those retrieved documents into the LLM's prompt, forcing it to answer based *only* on the provided context.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, run: !pip install -q langchain langchain-community sentence-transformers faiss-cpu

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate
import numpy as np

print("RAG libraries imported successfully!")

## Step 1: Data Ingestion and Chunking

LLMs have strict token limits (Context Windows), and embedding models struggle to capture the specific meaning of a 50-page document in a single vector. 

Therefore, we must **Chunk** the data. We use a `RecursiveCharacterTextSplitter`, which analytically attempts to split text at natural boundaries (like double newlines, then single newlines, then spaces) to keep semantic ideas together. We also include a **Chunk Overlap** so that a sentence cut in half retains its context in the next chunk.

In [ ]:
# Cell 4: Chunking the Knowledge Base

# 1. Our proprietary company data (The LLM does not know this!)
proprietary_text = """
Acme Corp was founded in 2015 by Jane Doe and John Smith. 
The flagship product is the Acme Rocket Skates, released in 2018, which utilize patented solid-fuel micro-thrusters. 
In 2023, Acme Corp acquired Zenith Dynamics to expand into the drone delivery market.
The current CEO is Jane Doe. The company headquarters is located in Austin, Texas.
Acme's Q3 revenue for 2023 was reported at $45 million, a 12% increase year-over-year.
"""

# 2. Configure the Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,       # Maximum characters per chunk
    chunk_overlap=20,     # Characters to overlap between chunks
    length_function=len
)

# 3. Execute the splitting
document_chunks = text_splitter.split_text(proprietary_text)

print(f"Original Text Length: {len(proprietary_text)} characters")
print(f"Created {len(document_chunks)} mathematical chunks.\n")

print("--- Analytical View of Chunks ---")
for i, chunk in enumerate(document_chunks):
    print(f"Chunk {i+1}: '{chunk}'")

## Step 2: Embedding and Vector Storage

Now we must translate human language into machine mathematics. We use an **Embedding Model** to map each chunk into a high-dimensional continuous vector space. 

If two chunks have similar meanings, their vectors will point in roughly the same direction. We store these vectors in a **Vector Database**. Here we use **FAISS** (Facebook AI Similarity Search), an incredibly fast library that optimizes nearest-neighbor search in high-dimensional spaces.

In [ ]:
# Cell 6: Generating Embeddings and Building the Vector Store

# 1. Initialize an open-source embedding model from HuggingFace
# 'all-MiniLM-L6-v2' maps sentences to a 384-dimensional dense vector space
print("Loading Embedding Model (Downloading weights if first time)...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Let's analytically prove what an embedding is:
sample_vector = embedding_model.embed_query("Test string")
print(f"A single text string becomes a vector of shape: {np.array(sample_vector).shape}")

# 2. Build the FAISS Vector Database using our chunks and the embedding model
print("\nIndexing chunks into FAISS Vector Database...")
vector_db = FAISS.from_texts(document_chunks, embedding_model)

print("Vector Database successfully built and indexed!")

## Step 3: Semantic Retrieval

When a user asks a question, we embed that query using the *exact same model*. 

We then perform a search against our database using **Cosine Similarity**, which measures the cosine of the angle between two vectors:
$$\text{Cosine Similarity} = \frac{A \cdot B}{||A|| \times ||B||}$$

The database returns the top $K$ chunks with the highest similarity scores.

In [ ]:
# Cell 8: Querying the Vector Database

user_query = "What is the company's most famous product and how does it work?"

# Perform similarity search, returning the top 2 most mathematically relevant chunks
k_retrieved = 2
retrieved_docs = vector_db.similarity_search_with_score(user_query, k=k_retrieved)

print(f"User Query: '{user_query}'\n")
print("--- Retrieved Context (Ranked by L2 Distance) ---")

# Lower score is better in FAISS (L2 distance is used by default under the hood)
retrieved_context = ""
for i, (doc, score) in enumerate(retrieved_docs):
    print(f"Rank {i+1} (Distance: {score:.4f}): {doc.page_content}")
    retrieved_context += doc.page_content + "\n"

## Step 4: Prompt Augmentation and Generation

This is the "Augmented Generation" part of RAG. We intercept the user's prompt before it reaches the LLM and completely rewrite it. 

We construct a **Prompt Template** that strictly instructs the LLM to act as an information synthesizer. We inject the `retrieved_context` and the `user_query`. By doing this, we strip the LLM of its tendency to hallucinate and force it to use only the provided facts.

In [ ]:
# Cell 10: Augmentation and Final Prompt Construction

# 1. Define the strict analytical prompt template
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a precise, analytical AI assistant. 
Use the following pieces of retrieved context to answer the question. 
If the answer is not contained within the context, explicitly state "I do not have sufficient information to answer this." Do not hallucinate.

Context:
{context}

User Question: 
{question}

Answer:"""
)

# 2. Construct the final augmented prompt
final_prompt = prompt_template.format(context=retrieved_context, question=user_query)

print("--- The Augmented Prompt Sent to the LLM ---")
print(final_prompt)

# --- LLM GENERATION SIMULATION ---
# In a real environment, you would pass this prompt to an OpenAI API or local LLM like this:
# llm = ChatOpenAI()
# response = llm.predict(final_prompt)

print("\n--- Simulated LLM Response ---")
print("Based on the provided context, Acme Corp's flagship product is the Acme Rocket Skates. They work by utilizing patented solid-fuel micro-thrusters.")

## Conclusion

You have successfully constructed a Retrieval-Augmented Generation architecture!

**Key Analytical Takeaways:**
1. **No Fine-Tuning Required:** Notice we did not change a single weight inside the LLM. We simply changed the *prompt* dynamically. This is vastly cheaper and faster than fine-tuning an LLM on company data.
2. **Traceability:** Because the LLM answers based on chunks retrieved from a database, you can always trace an answer back to its source document, completely eliminating the "black box" hallucination problem.
3. **The Bottleneck:** RAG is only as good as its retrieval phase. If your chunking strategy is poor, or your embedding model misses semantic nuances, the wrong context is retrieved, and the LLM will fail to answer the question.